# Notebook 05 -- Higher-Order Real (Figure 19)

This notebook loads the Thiers13 simplicial complex and runs higher-order
Hodge Laplacian renormalization to produce compression curves.

**Figure produced:**
- **Figure 19:** 1x3 panel -- compression curves for Thiers13 under Hodge,
  Bochner, and cross-order Laplacians.

In [ ]:
%matplotlib inline
import json
from pathlib import Path
from itertools import combinations
from collections import defaultdict
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from harmonic_morphisms import g_len, higher_order_H_CF_curves, fbc
from harmonic_morphisms.simplicial import laplacians, XO_laplacian, compute_entropic_C, compute_spectral_d

DATA_DIR = Path("../data")

SMALL_SIZE = 10; MEDIUM_SIZE = 12; BIGGER_SIZE = 13
plt.rc("font", size=SMALL_SIZE)
plt.rc("axes", titlesize=SMALL_SIZE)
plt.rc("axes", labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("legend", fontsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

## Load Thiers13 Simplicial Complex

In [ ]:
with open(DATA_DIR / "networks" / "random_10_0.85min_cliques_Thiers13.json", 'r') as f:
    doc = json.load(f)
simplices = doc[0]

# Extract nodes, edges, faces from simplices
nodes_set, edges_set, faces_set = set(), set(), set()
for simplex in simplices:
    for v in simplex:
        nodes_set.add(v)
    if len(simplex) == 2:
        edges_set.add(tuple(sorted(simplex)))
    elif len(simplex) == 3:
        face = tuple(sorted(simplex))
        faces_set.add(face)
        for edge in combinations(face, 2):
            edges_set.add(tuple(sorted(edge)))

# Remap to 0..N-1
sorted_nodes = sorted(nodes_set)
node_mapping = {old: new for new, old in enumerate(sorted_nodes)}
def remap(simplices_list):
    return [sorted([node_mapping[v] for v in s]) for s in simplices_list]

edges = np.array(remap(list(edges_set)))
faces = np.array(remap(list(faces_set)))
nodes = np.array([[i] for i in range(len(sorted_nodes))])

# Extract largest connected component of faces (face adjacency graph)
def build_face_adjacency(faces):
    edge_to_faces = defaultdict(set)
    for idx, face in enumerate(faces):
        for edge in combinations(sorted(face), 2):
            edge_to_faces[edge].add(idx)
    G = nx.Graph()
    G.add_nodes_from(range(len(faces)))
    for face_indices in edge_to_faces.values():
        face_list = list(face_indices)
        for i in range(len(face_list)):
            for j in range(i + 1, len(face_list)):
                G.add_edge(face_list[i], face_list[j])
    return G

G_adj = build_face_adjacency(faces)
largest_cc = max(nx.connected_components(G_adj), key=len)
faces = faces[list(largest_cc)]

sc = {
    'nodes': nodes, 'n0': len(nodes),
    'edges': edges, 'n1': len(edges),
    'faces': faces, 'n2': len(faces),
    'tetrahedra': np.zeros((0, 4), dtype=int), 'n3': 0,
    '4-simplices': np.zeros((0, 5), dtype=int), 'n4': 0,
}

print(f"Thiers13: {sc['n0']} nodes, {sc['n1']} edges, {sc['n2']} faces")

## Compute Laplacians and Harmonic Degree Curves

In [ ]:
L0, L1, L2, L3, L4, node_dict, edge_dict, face_dict, tet_dict = laplacians(sc)

# Bochner decomposition of L2
B2, F2 = fbc(L2)

# Cross-order Laplacian (faces diffusing via shared edges)
X2 = XO_laplacian(sc, 2, 1)

print(f"L2 shape: {L2.shape}, B2 shape: {B2.shape}, X2 shape: {X2.shape}")

In [ ]:
print("Running higher_order_H_CF_curves with Hodge L2...")
Ag_h, g_h, DEG_H_h, M_DEG_H_h, STD_H_h, AV_H_h, STD_V_H_h, NOT_H_h, \
    DEG_CF_h, M_DEG_CF_h, STD_CF_h, AV_CF_h, STD_V_CF_h, NOT_CF_h, \
    gV_h, t_span_h = higher_order_H_CF_curves(sc, dim=2, Lap=L2, n=100)

print("Running higher_order_H_CF_curves with Bochner B2...")
Ag_b, g_b, DEG_H_b, M_DEG_H_b, STD_H_b, AV_H_b, STD_V_H_b, NOT_H_b, \
    DEG_CF_b, M_DEG_CF_b, STD_CF_b, AV_CF_b, STD_V_CF_b, NOT_CF_b, \
    gV_b, t_span_b = higher_order_H_CF_curves(sc, dim=2, Lap=B2, n=100)

print("Running higher_order_H_CF_curves with XO X2...")
Ag_x, g_x, DEG_H_x, M_DEG_H_x, STD_H_x, AV_H_x, STD_V_H_x, NOT_H_x, \
    DEG_CF_x, M_DEG_CF_x, STD_CF_x, AV_CF_x, STD_V_CF_x, NOT_CF_x, \
    gV_x, t_span_x = higher_order_H_CF_curves(sc, dim=2, Lap=X2, n=100)

print("Done.")

## Figure 19 -- Thiers13 Compression Curves

In [ ]:
n_faces = sc["n2"]

comp_h = 1 - np.array(g_len(g_h)) / n_faces
comp_b = 1 - np.array(g_len(g_b)) / n_faces
comp_x = 1 - np.array(g_len(g_x)) / n_faces

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, name, comp, m_h, m_cf, std_h in [
    (axes[0], "Hodge L2", comp_h, M_DEG_H_h, M_DEG_CF_h, STD_H_h),
    (axes[1], "Bochner B2", comp_b, M_DEG_H_b, M_DEG_CF_b, STD_H_b),
    (axes[2], "XO X2", comp_x, M_DEG_H_x, M_DEG_CF_x, STD_H_x),
]:
    ax.plot(comp, m_h, "ob--", linewidth=1.5, markersize=4, label="H Mod.")
    ax.plot(comp, m_cf, "og--", linewidth=1.5, markersize=4, label="CF Mod.")
    ax.plot(comp, std_h, "ok--", linewidth=1.5, markersize=4, label="STD H")
    ax.set_xlabel("Compression")
    ax.set_ylabel("Harmonic degree")
    ax.set_ylim([-0.1, 1.2])
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Thiers13 Simplicial Complex")
plt.tight_layout()
plt.savefig("fig19_thiers13.pdf", bbox_inches="tight")
plt.show()
print("Saved fig19_thiers13.pdf")